# Gradient-Based Interpretability

While activation patching uses causal interventions, **gradient-based methods** use the model's derivatives to measure how sensitive the output is to each input. This is complementary to patching — gradients tell you about local sensitivity, while patching tells you about causal importance.

This notebook covers:
1. **Gradient × Input**: Simple, fast attribution
2. **Integrated Gradients**: Path-integral attribution (satisfies completeness)
3. **Logit Gradient Attribution**: Which components affect a logit?
4. **Comparing gradient-based and patching-based methods**

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt

import irtk
from irtk.gradients import (
    gradient_x_input, gradient_norm, integrated_gradients,
    integrated_gradients_full, logit_gradient_attribution,
    token_attribution,
)
from irtk.circuits import direct_logit_attribution, residual_stream_attribution
from irtk import vis

print("Loading GPT-2 Small...")
model = irtk.HookedTransformer.from_pretrained("gpt2")
tokenizer = model.tokenizer
print(f"Loaded: {model.cfg.n_layers} layers, {model.cfg.n_heads} heads")

## 1. Gradient × Input Attribution

The simplest gradient method: compute `grad(logit) * embedding` for each input token. The norm tells you how much each token "contributes" to the prediction.

This is fast (one backward pass) but only measures local sensitivity.

In [ ]:
prompt = "The capital of France is"
tokens = model.to_tokens(prompt)
logits = model(tokens)

pred_token = int(jnp.argmax(logits[-1]))
pred_str = tokenizer.decode([pred_token])
print(f"Prompt: {prompt!r}")
print(f"Prediction: {pred_str!r}")

# Gradient × input attribution
attr_gxi = gradient_x_input(model, tokens, target_token=pred_token)
# Gradient norm attribution
attr_gn = gradient_norm(model, tokens, target_token=pred_token)

token_labels = [tokenizer.decode([t]) for t in np.array(tokens)]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

ax1.bar(range(len(attr_gxi)), attr_gxi)
ax1.set_xticks(range(len(token_labels)))
ax1.set_xticklabels(token_labels, rotation=45, ha='right')
ax1.set_title(f"Gradient × Input for {pred_str!r}")
ax1.set_ylabel("Attribution")
ax1.grid(True, alpha=0.3)

ax2.bar(range(len(attr_gn)), attr_gn)
ax2.set_xticks(range(len(token_labels)))
ax2.set_xticklabels(token_labels, rotation=45, ha='right')
ax2.set_title(f"Gradient Norm for {pred_str!r}")
ax2.set_ylabel("Gradient Norm")
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nPer-token attributions (grad × input):")
for label, val in zip(token_labels, attr_gxi):
    print(f"  {label:>10s}: {val:.4f}")

## 2. Integrated Gradients

**Integrated Gradients** (Sundararajan et al. 2017) addresses a key limitation of gradient × input: it satisfies the **completeness axiom** — attributions sum to the output difference.

Instead of computing the gradient at the actual input, it integrates the gradient along a path from a baseline (zero embedding) to the actual input:

$$\text{IG}(x) = (x - x') \cdot \int_0^1 \nabla f(x' + t(x-x')) \, dt$$

This is more principled but slower (requires ~50 forward passes).

In [ ]:
# Integrated gradients with different step counts
attr_ig10 = integrated_gradients(model, tokens, pred_token, n_steps=10)
attr_ig50 = integrated_gradients(model, tokens, pred_token, n_steps=50)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

x = range(len(tokens))
ax1.bar(x, attr_ig10, alpha=0.7, label='10 steps')
ax1.bar(x, attr_ig50, alpha=0.7, label='50 steps')
ax1.set_xticks(x)
ax1.set_xticklabels(token_labels, rotation=45, ha='right')
ax1.set_title(f"Integrated Gradients for {pred_str!r}")
ax1.set_ylabel("Attribution")
ax1.legend()
ax1.grid(True, alpha=0.3)

# Compare all three methods
methods = {
    'Grad × Input': attr_gxi / (attr_gxi.max() + 1e-8),
    'Grad Norm': attr_gn / (attr_gn.max() + 1e-8),
    'Integrated Grad': attr_ig50 / (attr_ig50.max() + 1e-8),
}

width = 0.25
for i, (name, vals) in enumerate(methods.items()):
    ax2.bar(np.arange(len(vals)) + i * width, vals, width, label=name)
ax2.set_xticks(np.arange(len(token_labels)) + width)
ax2.set_xticklabels(token_labels, rotation=45, ha='right')
ax2.set_title("All Methods (normalized)")
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Full integrated gradients: see which embedding dimensions matter most
full_ig = integrated_gradients_full(model, tokens, pred_token, n_steps=30)
print(f"Full IG shape: {full_ig.shape}  (seq_len × d_model)")

fig, ax = plt.subplots(figsize=(12, 4))
im = ax.imshow(full_ig.T, aspect='auto', cmap='RdBu_r',
               vmin=-np.abs(full_ig).max(), vmax=np.abs(full_ig).max())
ax.set_xlabel("Token Position")
ax.set_ylabel("Embedding Dimension")
ax.set_xticks(range(len(token_labels)))
ax.set_xticklabels(token_labels, rotation=45, ha='right')
ax.set_title(f"Full Integrated Gradients for {pred_str!r}")
plt.colorbar(im, ax=ax, label='Attribution')
plt.tight_layout()
plt.show()

## 3. Logit Gradient Attribution

Beyond per-token attribution, we can measure how much each **model component** (embedding, attention heads, MLPs) contributes to a logit.

This uses the gradient of the logit w.r.t. the final residual stream, accounting for LayerNorm.

In [ ]:
# Gradient-based component attribution
grad_attrs = logit_gradient_attribution(model, tokens, target_token=pred_token)

# Compare with direct logit attribution (cache-based)
_, cache = model.run_with_cache(tokens)
cache_attrs = residual_stream_attribution(model, cache, token=pred_token)

# Plot both side by side
names = list(grad_attrs.keys())
grad_values = [float(grad_attrs[n]) for n in names]
cache_values = [float(cache_attrs.get(n, 0.0)) for n in names]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8))

ax1.barh(range(len(names)), grad_values)
ax1.set_yticks(range(len(names)))
ax1.set_yticklabels(names, fontsize=8)
ax1.set_title(f"Gradient-Based Component Attribution for {pred_str!r}")
ax1.set_xlabel("Attribution")
ax1.grid(True, alpha=0.3)

ax2.barh(range(len(names)), cache_values)
ax2.set_yticks(range(len(names)))
ax2.set_yticklabels(names, fontsize=8)
ax2.set_title(f"Cache-Based Component Attribution (no LN gradient) for {pred_str!r}")
ax2.set_xlabel("Attribution")
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Gradient-based attribution accounts for LayerNorm's effect on the gradient,")
print("while direct logit attribution projects through W_U directly.")

## 4. Unified Token Attribution Interface

The `token_attribution()` function provides a unified interface for all three gradient methods.

In [ ]:
# Try on a different prompt
prompt2 = "John went to the store. He bought some milk. John drank the"
tokens2 = model.to_tokens(prompt2)
logits2 = model(tokens2)
pred2 = int(jnp.argmax(logits2[-1]))
pred2_str = tokenizer.decode([pred2])
labels2 = [tokenizer.decode([t]) for t in np.array(tokens2)]

print(f"Prompt: {prompt2!r}")
print(f"Prediction: {pred2_str!r}")

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, method in zip(axes, ['grad_x_input', 'grad_norm', 'integrated_gradients']):
    if method == 'integrated_gradients':
        attr = token_attribution(model, tokens2, pred2, method=method, n_steps=20)
    else:
        attr = token_attribution(model, tokens2, pred2, method=method)
    
    ax.bar(range(len(attr)), attr)
    ax.set_xticks(range(len(labels2)))
    ax.set_xticklabels(labels2, rotation=90, fontsize=7)
    ax.set_title(method.replace('_', ' ').title())
    ax.set_ylabel("Attribution")
    ax.grid(True, alpha=0.3)

plt.suptitle(f"Token Attribution for predicting {pred2_str!r}", fontsize=14)
plt.tight_layout()
plt.show()

## Key Takeaways

1. **Gradient × Input** is fast (one backward pass) but only measures local sensitivity. Good for quick screening.

2. **Integrated Gradients** is more principled — it satisfies the completeness axiom (attributions sum to the output difference). It needs ~50 steps for convergence.

3. **Logit Gradient Attribution** measures component-level sensitivity accounting for LayerNorm's non-linear effect on gradients.

4. Gradient methods and patching methods are **complementary**:
   - Gradients tell you about local sensitivity ("how would an infinitesimal change here affect the output?")
   - Patching tells you about causal importance ("does this component actually contain the information?")

### API Reference

```python
from irtk.gradients import (
    gradient_x_input,          # [seq_len] grad * embed norm per position
    gradient_norm,             # [seq_len] gradient norm per position
    integrated_gradients,      # [seq_len] IG attribution per position
    integrated_gradients_full, # [seq_len, d_model] full IG matrix
    logit_gradient_attribution,# component -> scalar attribution
    input_jacobian,            # [d_vocab, seq_len, d_model] full Jacobian
    token_attribution,         # unified interface for all methods
)
```